# IVF 1-Level vs 2-Level (Quick Experiment)\n
This notebook compares retrieval quality + latency between single-level IVF and two-level IVF using the ADB-project logic.\n
It is intentionally simple and practical.

In [ ]:
import os\n
import time\n
import shutil\n
import importlib\n
from pathlib import Path\n
import numpy as np

In [ ]:
# point to ADB project\n
ADB_ROOT = Path(r"g:/Engineering/Database/ADB-project")\n
WORK_ROOT = Path(r"g:/Engineering/GP/Docs/Expirements/ivf_compare_work")\n
WORK_ROOT.mkdir(parents=True, exist_ok=True)\n
\n
import sys\n
if str(ADB_ROOT) not in sys.path:\n
    sys.path.insert(0, str(ADB_ROOT))\n
\n
import constants as adb_constants\n
from vec_db import VecDB

In [ ]:
# synthetic dataset (use real embeddings later if you want)\n
SEED = 7\n
DIM = 64\n
DB_SIZE = 20000\n
N_QUERIES = 40\n
TOP_K = 10\n
\n
rng = np.random.default_rng(SEED)\n
db_vectors = rng.normal(size=(DB_SIZE, DIM)).astype(np.float32)\n
db_vectors /= (np.linalg.norm(db_vectors, axis=1, keepdims=True) + 1e-12)\n
\n
q_idx = rng.choice(DB_SIZE, size=N_QUERIES, replace=False)\n
queries = db_vectors[q_idx].copy()

In [ ]:
def brute_force_ids(base, query, top_k):\n
    sims = base @ query\n
    idx = np.argpartition(sims, -top_k)[-top_k:]\n
    idx = idx[np.argsort(sims[idx])[::-1]]\n
    return idx.tolist()\n
\n
def build_and_eval(two_level: bool):\n
    mode = "two" if two_level else "one"\n
    db_file = WORK_ROOT / f"db_{mode}.dat"\n
    idx_file = WORK_ROOT / f"idx_{mode}.dat"\n
    idx_folder = WORK_ROOT / f"idx_{mode}_index"\n
\n
    if db_file.exists():\n
        db_file.unlink()\n
    if idx_file.exists():\n
        idx_file.unlink()\n
    if idx_folder.exists():\n
        shutil.rmtree(idx_folder)\n
\n
    adb_constants.DIMENSION = DIM\n
    adb_constants.USE_TWO_LEVEL_IVF = two_level\n
\n
    db_vectors.tofile(db_file)\n
\n
    db = VecDB(database_file_path=str(db_file), index_file_path=str(idx_file), new_db=False)\n
\n
    t0 = time.perf_counter()\n
    db.public_build_index()\n
    build_time = time.perf_counter() - t0\n
\n
    total_recall = 0.0\n
    t1 = time.perf_counter()\n
    for q in queries:\n
        pred = db.retrieve(q.reshape(1, -1), top_k=TOP_K)\n
        gt = brute_force_ids(db_vectors, q, TOP_K)\n
        hit = len(set(pred).intersection(set(gt)))\n
        total_recall += hit / TOP_K\n
    query_time = time.perf_counter() - t1\n
\n
    return {\n
        "mode": "2-level IVF" if two_level else "1-level IVF",\n
        "build_sec": build_time,\n
        "query_sec_total": query_time,\n
        "query_ms_avg": (query_time / N_QUERIES) * 1000.0,\n
        "recall_at_k": total_recall / N_QUERIES,\n
    }

In [ ]:
res_one = build_and_eval(two_level=False)\n
res_two = build_and_eval(two_level=True)\n
res_one, res_two

In [ ]:
import pandas as pd\n
df = pd.DataFrame([res_one, res_two])\n
df

## Notes\n
- For your real pipeline, use your actual embeddings instead of synthetic vectors.\n
- This test is still useful to check tradeoff trend (speed vs recall).\n
- If 2-level is better enough for your recall target, keep it as the production default.